# 🏛️ POC 11: Half-Century Macro Historical Stress-Testing (1978–2026)

**File**: [`research/notebooks/algo-alpha-execution/11_half_century_macro_backtest_1975_2026.ipynb`](file:///c:/Users/honza/Desktop/projects/stock-analysis/research/notebooks/algo-alpha-execution/11_half_century_macro_backtest_1975_2026.ipynb)  
**Historical Backtest Horizon**: **January 1978 - August 2026 (48.6 Years / 12,200+ Daily Trading Sessions)**  
**Continuous Multi-Decade Data Ingestion**: 50 years of OHLC daily price bars, expanding walk-forward ML models across 49 annual walk-forward cycles, and continuous GDELT news tone & NLP sentiment memory decay.

---

### Executive Summary & Quantitative Scope
To rigorously test whether our machine learning alpha signals, trailing volatility stops, and macro crash filters remain structurally invariant across half a century of regime shifts, we stress-test our models across **10 major historical crisis epochs**:

```
┌────────────────────────────────────────────────────────────────────────────────────────┐
│ 50 YEARS OF HISTORICAL REGIMES TESTED (1978–2026 / 12,200+ TRADING SESSIONS)           │
│ 1. 1979-1982 Volcker 20% Fed Rate Shock & Double-Dip Stagflation                       │
│ 2. 1987 Black Monday Crash (-22.6% in a Single Day)                                    │
│ 3. 1990 Gulf War Oil Spike & Recession                                                 │
│ 4. 1997-1998 Asian Contagion & Russian LTCM Debt Default                               │
│ 5. 2000-2002 Dot-Com Bubble Collapse (-49% SPX, -78% Tech)                            │
│ 6. 2007-2009 Global Financial Crisis (-55% S&P 500)                                   │
│ 7. 2011 US Sovereign Debt Downgrade & European Debt Crisis                             │
│ 8. 2018 Volmageddon & Fed Quantitative Tightening Sell-off                             │
│ 9. 2020 COVID-19 Flash Crash (-34% in 23 Days)                                         │
│ 10. 2022 Inflation & 500bps Rate-Hike Bear Market                                      │
└────────────────────────────────────────────────────────────────────────────────────────┘
```

**Strategies & Benchmarks Evaluated**:
1. 📰 **Unified Engine + Daily News FinBERT (Top 10 Active)**: Confluence Dynamic Sizing (5%–20%) + Trailing ATR Stops ($2.5 \cdot \text{ATR}_{14}$) + Macro Vol Guard ($2.0\sigma$).
2. 🚀 **Baseline Naive XGBoost + Daily News FinBERT (Top 100)**: Active Top 100 equal-weighted model augmented with daily news FinBERT features.
3. 🤖 **Baseline Naive XGBoost Standard (Top 100 - No News)**: Baseline fundamental/TA model without daily news.
4. 📈 **S&P 500 Index (`^GSPC` Benchmark)**: The official broad market benchmark across 1978–2026.

## 1. Setup, Configuration & Dependencies

In [1]:
import os
import sys
import datetime
from datetime import timedelta
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import yfinance as yf
from tqdm.auto import tqdm

# Robust project root discovery
current_dir = os.path.abspath(os.getcwd())
while current_dir and not os.path.exists(os.path.join(current_dir, "src")):
    parent = os.path.dirname(current_dir)
    if parent == current_dir:
        break
    current_dir = parent

PROJECT_ROOT = current_dir
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config import DATA_DIR, INITIAL_CAPITAL

LOCAL_DATA_DIR = os.path.join(PROJECT_ROOT, "research", "notebooks", "algo-alpha-execution", "data", "fetched")
if not os.path.exists(LOCAL_DATA_DIR):
    LOCAL_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "fetched")

print(f"📁 Project Root: {PROJECT_ROOT}")
print(f"📁 Local Data Directory: {LOCAL_DATA_DIR}")
print(f"💰 Initial Capital: ${INITIAL_CAPITAL}")
print(f"⏳ Backtest Scope: 1978-2026 (48.6 Years / 12,200+ Trading Days)")

📁 Project Root: c:\Users\honza\Desktop\projects\stock-analysis
📁 Local Data Directory: c:\Users\honza\Desktop\projects\stock-analysis\research\notebooks\algo-alpha-execution\data\fetched
💰 Initial Capital: $100.0
⏳ Backtest Scope: 1978-2026 (48.6 Years / 12,200+ Trading Days)


## 2. Ingesting 50-Year Market Data & Walk-Forward Predictions

In [2]:
def load_50year_dataset():
    preds_path = os.path.join(LOCAL_DATA_DIR, "deep_historical_1975_2026_predictions_poc.xlsx")
    df_preds = pd.read_excel(preds_path)
    df_preds['date'] = pd.to_datetime(df_preds['date'])
    all_tickers = sorted(df_preds['ticker'].unique())
    print(f"✅ Loaded {len(df_preds)} prediction records across {len(all_tickers)} tickers over 1978-2026!")
    
    unique_tickers = all_tickers + ['^GSPC']
    min_date = (df_preds['date'].min() - timedelta(days=60)).strftime('%Y-%m-%d')
    max_date = (df_preds['date'].max() + timedelta(days=10)).strftime('%Y-%m-%d')
    
    print(f"📈 Downloading OHLC market data from 1977 to 2026 for {len(unique_tickers)} tickers...")
    ohlc = yf.download(unique_tickers, start=min_date, end=max_date, auto_adjust=True, progress=False)
    
    close_p = ohlc['Close']
    high_p = ohlc['High']
    low_p = ohlc['Low']
    
    close_p.index = pd.to_datetime(close_p.index).tz_localize(None)
    high_p.index = pd.to_datetime(high_p.index).tz_localize(None)
    low_p.index = pd.to_datetime(low_p.index).tz_localize(None)
    
    # Compute ATR(14)
    atr_dict = {}
    for t in all_tickers:
        if t in close_p.columns and t in high_p.columns and t in low_p.columns:
            c = close_p[t]
            h = high_p[t]
            l = low_p[t]
            prev_c = c.shift(1)
            tr = pd.concat([h - l, (h - prev_c).abs(), (l - prev_c).abs()], axis=1).max(axis=1)
            atr_dict[t] = tr.ewm(alpha=1/14, adjust=False).mean()
    df_atr = pd.DataFrame(atr_dict)
    
    # Macro Volatility Filter on S&P 500 (^GSPC)
    spx_col = '^GSPC' if '^GSPC' in close_p.columns else close_p.columns[-1]
    spx_ret = close_p[spx_col].pct_change()
    ewma_lam = 1.0 - (2.0 / 21.0)
    spx_ewma_var = (spx_ret**2).ewm(alpha=(1 - ewma_lam), adjust=False).mean()
    spx_ewma_vol = np.sqrt(spx_ewma_var) * np.sqrt(252)
    spx_vol_ma = spx_ewma_vol.rolling(60).mean()
    spx_vol_std = spx_ewma_vol.rolling(60).std()
    spx_vol_zscore = (spx_ewma_vol - spx_vol_ma) / (spx_vol_std + 1e-9)
    
    return df_preds, close_p, df_atr, spx_vol_zscore, all_tickers, spx_col

df_predictions, close_prices, df_atr_matrix, macro_vol_z, universe_tickers, spx_ticker = load_50year_dataset()
all_sim_dates = sorted(list(set(df_predictions['date'].unique()) & set(close_prices.index)))
print(f"✅ Total 50-Year Simulation Trading Days: {len(all_sim_dates)} ({all_sim_dates[0].strftime('%Y-%m-%d')} to {all_sim_dates[-1].strftime('%Y-%m-%d')})")

✅ Loaded 638434 prediction records across 60 tickers over 1978-2026!
📈 Downloading OHLC market data from 1977 to 2026 for 61 tickers...


✅ Total 50-Year Simulation Trading Days: 12264 (1978-01-03 to 2026-08-27)


C:\Users\honza\AppData\Local\Temp\ipykernel_7968\2632210548.py:37: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  spx_ret = close_p[spx_col].pct_change()


## 3. Parametric Strategy Simulation across Half a Century (1978–2026)

In [3]:
def simulate_unified_news_engine(
    preds_df, prices_df, atr_df, macro_z, all_dates,
    base_cap=0.08, max_confluence_cap=0.20, atr_multiplier=2.5, rebalance_days=25, z_threshold=2.0, active_n=10, transaction_cost_bps=15
):
    fee_rate = transaction_cost_bps / 10000.0
    cash = INITIAL_CAPITAL
    active_positions = {}
    history = []
    days_since_rebalance = rebalance_days
    
    for d in all_dates:
        p_now = prices_df.loc[d]
        atr_now = atr_df.loc[d] if d in atr_df.index else None
        z_curr = macro_z.loc[d] if d in macro_z.index else 0.0
        
        stopped_out = []
        for t, pos in list(active_positions.items()):
            if t in p_now and pd.notna(p_now[t]):
                price_curr = p_now[t]
                atr_curr = atr_now[t] if (atr_now is not None and t in atr_now and pd.notna(atr_now[t])) else (price_curr * 0.03)
                if price_curr > pos['highest_price']:
                    pos['highest_price'] = price_curr
                    pos['stop_price'] = max(pos['stop_price'], price_curr - (atr_multiplier * atr_curr))
                if price_curr <= pos['stop_price']:
                    cash += pos['shares'] * price_curr * (1.0 - fee_rate)
                    stopped_out.append(t)
        for t in stopped_out:
            del active_positions[t]
            
        if days_since_rebalance >= rebalance_days:
            days_since_rebalance = 0
            is_vol_spike = (pd.notna(z_curr) and z_curr > z_threshold)
            cash_buffer_ratio = 0.30 if is_vol_spike else 0.0
            
            day_preds = preds_df[preds_df['date'] == d].copy()
            pos_preds = day_preds[day_preds['predicted_return_news_multimodal'] > 0.0]
            if pos_preds.empty:
                pos_preds = day_preds
                
            if not pos_preds.empty:
                selected = pos_preds.sort_values('predicted_return_news_multimodal', ascending=False).head(active_n)
                conf_score = selected['confluence_score'] if 'confluence_score' in selected.columns else 0.0
                caps = base_cap + (conf_score / 6.0) * (max_confluence_cap - base_cap)
                caps = caps.clip(lower=base_cap, upper=max_confluence_cap)
                inv_vols = 1.0 / selected['ewma_volatility'].clip(lower=0.05)
                raw_weights = inv_vols / inv_vols.sum()
                bounded_weights = np.minimum(raw_weights, caps)
                final_weights = (bounded_weights / bounded_weights.sum()) * (1.0 - cash_buffer_ratio)
                target_alloc = dict(zip(selected['ticker'], final_weights))
            else:
                target_alloc = {}
                
            for t in list(active_positions.keys()):
                if t not in target_alloc:
                    if t in p_now and pd.notna(p_now[t]):
                        cash += active_positions[t]['shares'] * p_now[t] * (1.0 - fee_rate)
                    del active_positions[t]
                    
            total_fund = cash + sum(pos['shares'] * p_now[t] for t, pos in active_positions.items() if t in p_now)
            cash = total_fund * cash_buffer_ratio
            investable = total_fund * (1.0 - cash_buffer_ratio)
            
            for t, w in target_alloc.items():
                if t in p_now and pd.notna(p_now[t]) and p_now[t] > 0:
                    price_curr = p_now[t]
                    atr_curr = atr_now[t] if (atr_now is not None and t in atr_now and pd.notna(atr_now[t])) else (price_curr * 0.03)
                    shares = (investable * (w / (1.0 - cash_buffer_ratio + 1e-9)) * (1.0 - fee_rate)) / price_curr
                    active_positions[t] = {
                        'shares': shares,
                        'entry_price': price_curr,
                        'highest_price': price_curr,
                        'stop_price': price_curr - (atr_multiplier * atr_curr),
                        'weight': w
                    }
                    
        days_since_rebalance += 1
        portfolio_val = cash + sum(pos['shares'] * p_now[t] for t, pos in active_positions.items() if t in p_now and pd.notna(p_now[t]))
        history.append({'date': d, 'portfolio_value': portfolio_val})
        
    return pd.DataFrame(history)

def simulate_baseline_news_top100_strategy(
    preds_df, prices_df, all_dates,
    rebalance_days=25, active_n=100, transaction_cost_bps=15
):
    # Baseline XGBoost augmented with Daily News FinBERT signals (Top 100 Equal-Weighted)
    fee_rate = transaction_cost_bps / 10000.0
    cash = INITIAL_CAPITAL
    active_positions = {}
    history = []
    days_since_rebalance = rebalance_days
    
    for d in all_dates:
        p_now = prices_df.loc[d]
        
        if days_since_rebalance >= rebalance_days:
            days_since_rebalance = 0
            day_preds = preds_df[preds_df['date'] == d].copy()
            pos_preds = day_preds[day_preds['predicted_return_news_multimodal'] > 0.0]
            if pos_preds.empty:
                pos_preds = day_preds
                
            if not pos_preds.empty:
                selected = pos_preds.sort_values('predicted_return_news_multimodal', ascending=False).head(active_n)
                weights = [1.0 / len(selected)] * len(selected)
                target_alloc = dict(zip(selected['ticker'], weights))
            else:
                target_alloc = {}
                
            for t in list(active_positions.keys()):
                if t not in target_alloc:
                    if t in p_now and pd.notna(p_now[t]):
                        cash += active_positions[t]['shares'] * p_now[t] * (1.0 - fee_rate)
                    del active_positions[t]
                    
            total_fund = cash + sum(pos['shares'] * p_now[t] for t, pos in active_positions.items() if t in p_now)
            cash = 0.0
            
            for t, w in target_alloc.items():
                if t in p_now and pd.notna(p_now[t]) and p_now[t] > 0:
                    shares = (total_fund * w * (1.0 - fee_rate)) / p_now[t]
                    active_positions[t] = {'shares': shares, 'entry_price': p_now[t], 'weight': w}
                    
        days_since_rebalance += 1
        portfolio_val = cash + sum(pos['shares'] * p_now[t] for t, pos in active_positions.items() if t in p_now and pd.notna(p_now[t]))
        history.append({'date': d, 'portfolio_value': portfolio_val})
        
    return pd.DataFrame(history)

def simulate_baseline_standard_top100_strategy(
    preds_df, prices_df, all_dates,
    rebalance_days=25, active_n=100, transaction_cost_bps=15
):
    # Standard Baseline XGBoost without Daily News signals (Top 100 Equal-Weighted)
    fee_rate = transaction_cost_bps / 10000.0
    cash = INITIAL_CAPITAL
    active_positions = {}
    history = []
    days_since_rebalance = rebalance_days
    
    for d in all_dates:
        p_now = prices_df.loc[d]
        
        if days_since_rebalance >= rebalance_days:
            days_since_rebalance = 0
            day_preds = preds_df[preds_df['date'] == d].copy()
            pos_preds = day_preds[day_preds['predicted_return_baseline'] > 0.0]
            if pos_preds.empty:
                pos_preds = day_preds
                
            if not pos_preds.empty:
                selected = pos_preds.sort_values('predicted_return_baseline', ascending=False).head(active_n)
                weights = [1.0 / len(selected)] * len(selected)
                target_alloc = dict(zip(selected['ticker'], weights))
            else:
                target_alloc = {}
                
            for t in list(active_positions.keys()):
                if t not in target_alloc:
                    if t in p_now and pd.notna(p_now[t]):
                        cash += active_positions[t]['shares'] * p_now[t] * (1.0 - fee_rate)
                    del active_positions[t]
                    
            total_fund = cash + sum(pos['shares'] * p_now[t] for t, pos in active_positions.items() if t in p_now)
            cash = 0.0
            
            for t, w in target_alloc.items():
                if t in p_now and pd.notna(p_now[t]) and p_now[t] > 0:
                    shares = (total_fund * w * (1.0 - fee_rate)) / p_now[t]
                    active_positions[t] = {'shares': shares, 'entry_price': p_now[t], 'weight': w}
                    
        days_since_rebalance += 1
        portfolio_val = cash + sum(pos['shares'] * p_now[t] for t, pos in active_positions.items() if t in p_now and pd.notna(p_now[t]))
        history.append({'date': d, 'portfolio_value': portfolio_val})
        
    return pd.DataFrame(history)

print("🚀 Running 50-Year Macro Historical Backtest (1978-2026 / 12,200+ Days)...")

# 1. Unified Engine + Daily News FinBERT (Top 10 Active)
df_strat_unified_news = simulate_unified_news_engine(df_predictions, close_prices, df_atr_matrix, macro_vol_z, all_sim_dates, base_cap=0.08, max_confluence_cap=0.20, rebalance_days=25, active_n=10)

# 2. Baseline Naive XGBoost + Daily News FinBERT (Top 100)
df_strat_base_news = simulate_baseline_news_top100_strategy(df_predictions, close_prices, all_sim_dates, rebalance_days=25, active_n=100)

# 3. Baseline Naive XGBoost Standard (Top 100 - No Daily News)
df_strat_base_std = simulate_baseline_standard_top100_strategy(df_predictions, close_prices, all_sim_dates, rebalance_days=25, active_n=100)

# 4. S&P 500 Benchmark (^GSPC)
sim_dates = df_strat_unified_news['date']
spx_prices = close_prices[spx_ticker].loc[close_prices.index.isin(sim_dates)]
spx_norm = (spx_prices / spx_prices.iloc[0]) * INITIAL_CAPITAL

df_master_eval = pd.DataFrame({
    'date': sim_dates,
    'Strategy_Unified_News_FinBERT_Top10': df_strat_unified_news['portfolio_value'].values,
    'Strategy_Baseline_News_FinBERT_Top100': df_strat_base_news['portfolio_value'].values,
    'Strategy_Baseline_Standard_Top100': df_strat_base_std['portfolio_value'].values,
    'Benchmark_SP500_Index': spx_norm.values
})

df_master_eval.head(10)

🚀 Running 50-Year Macro Historical Backtest (1978-2026 / 12,200+ Days)...


,date,Strategy_Unified_News_FinBERT_Top10,Strategy_Baseline_News_FinBERT_Top100,Strategy_Baseline_Standard_Top100,Benchmark_SP500_Index
0,1978-01-03,99.850000,99.850000,99.850000,100.000000
1,1978-01-04,99.835590,99.655809,99.494151,99.680236
2,1978-01-05,99.124403,99.010903,98.685029,98.848858
3,1978-01-06,96.940192,97.096429,97.156063,97.655087
4,1978-01-09,96.828697,96.377322,96.181431,96.610530
5,1978-01-10,96.356137,95.796564,95.473198,96.109570
6,1978-01-11,96.138986,95.104561,95.032570,95.651245
7,1978-01-12,96.362905,95.550414,95.171170,95.736517
8,1978-01-13,96.441757,95.406129,95.066566,95.597956
9,1978-01-16,96.287476,94.934180,94.791442,95.320828


## 4. Quantitative Analytics & Half-Century Performance Matrix (1978–2026)

In [4]:
def compute_strategy_analytics(series, spx_series, rf=0.035):
    daily_rets = series.pct_change().dropna()
    spx_rets = spx_series.pct_change().dropna()
    aligned = pd.concat([daily_rets, spx_rets], axis=1, join='inner').dropna()
    r_strat, r_spx = aligned.iloc[:, 0], aligned.iloc[:, 1]
    
    n_years = len(r_strat) / 252.0
    total_ret = (series.iloc[-1] / series.iloc[0]) - 1.0
    cagr = (series.iloc[-1] / series.iloc[0]) ** (1.0 / max(n_years, 0.1)) - 1.0
    ann_excess = (r_strat.mean() * 252.0) - rf
    ann_vol = r_strat.std() * np.sqrt(252.0)
    sharpe = ann_excess / ann_vol if ann_vol > 0 else 0.0
    downside_vol = r_strat[r_strat < 0].std() * np.sqrt(252.0)
    sortino = ann_excess / downside_vol if downside_vol > 0 else 0.0
    
    drawdown = (series - series.cummax()) / series.cummax()
    max_dd = drawdown.min()
    calmar = cagr / abs(max_dd) if abs(max_dd) > 0 else 0.0
    cov_matrix = np.cov(r_strat, r_spx)
    beta = cov_matrix[0, 1] / cov_matrix[1, 1] if cov_matrix[1, 1] > 0 else 1.0
    alpha = (cagr - rf) - beta * (((spx_series.iloc[-1] / spx_series.iloc[0]) ** (1.0 / max(n_years, 0.1)) - 1.0) - rf)
    
    return {
        'Total Return (%)': total_ret * 100.0,
        'CAGR (%)': cagr * 100.0,
        'Sharpe Ratio': sharpe,
        'Sortino Ratio': sortino,
        'Max Drawdown (%)': max_dd * 100.0,
        'Calmar Ratio': calmar,
        'Market Beta (β)': beta,
        'Jensen Alpha (α %)': alpha * 100.0
    }

eval_list = [
    ('Unified Engine + Daily News FinBERT (Top 10 Active)', df_master_eval['Strategy_Unified_News_FinBERT_Top10']),
    ('Baseline Naive XGBoost + Daily News FinBERT (Top 100)', df_master_eval['Strategy_Baseline_News_FinBERT_Top100']),
    ('Baseline Naive XGBoost Standard (Top 100 - No News)', df_master_eval['Strategy_Baseline_Standard_Top100']),
    ('S&P 500 Index (^GSPC Benchmark)', df_master_eval['Benchmark_SP500_Index'])
]

summary_stats = []
for name, s in eval_list:
    summary_stats.append({'Strategy / Model': name, **compute_strategy_analytics(s, df_master_eval['Benchmark_SP500_Index'])})

df_analytics_table = pd.DataFrame(summary_stats)
print("=== 50-YEAR (1978-2026) PERFORMANCE & RISK MATRIX ===")
df_analytics_table

=== 50-YEAR (1978-2026) PERFORMANCE & RISK MATRIX ===


,Strategy / Model,Total Return (%),CAGR (%),Sharpe Ratio,Sortino Ratio,Max Drawdown (%),Calmar Ratio,Market Beta (β),Jensen Alpha (α %)
0,Unified Engine + Daily News FinBERT (Top 10 Ac...,20992.226110,11.624579,0.518730,0.705011,-44.843575,0.259225,0.683713,4.029589
1,Baseline Naive XGBoost + Daily News FinBERT (T...,185620.814184,16.727678,0.774525,1.007921,-47.750352,0.350315,0.955081,7.507375
2,Baseline Naive XGBoost Standard (Top 100 - No ...,187330.579341,16.749662,0.776406,1.008807,-47.913118,0.349584,0.954412,7.533365
3,S&P 500 Index (^GSPC Benchmark),8140.236900,9.489339,0.404929,0.511454,-56.775388,0.167138,1.000000,0.000000


## 5. Stress-Testing Across 10 Historical Crisis Epochs (1978–2026)

In [5]:
crises = [
    ("1. 1979-1982 Volcker 20% Fed Rate Shock", "1979-01-02", "1982-08-12"),
    ("2. 1987 Black Monday Crash", "1987-10-01", "1987-11-30"),
    ("3. 1990 Gulf War Recession & Oil Shock", "1990-07-16", "1990-10-11"),
    ("4. 1997-1998 Asian Crisis & LTCM Collapse", "1997-07-01", "1998-10-08"),
    ("5. 2000-2002 Dot-Com Bubble Collapse", "2000-03-24", "2002-10-09"),
    ("6. 2007-2009 Global Financial Crisis (GFC)", "2007-10-09", "2009-03-09"),
    ("7. 2011 US Sovereign Debt Downgrade", "2011-05-02", "2011-10-03"),
    ("8. 2018 Volmageddon & Q4 Sell-off", "2018-09-20", "2018-12-24"),
    ("9. 2020 COVID-19 Flash Crash", "2020-02-19", "2020-03-23"),
    ("10. 2022 Inflation & Rate-Hike Bear Market", "2022-01-03", "2022-10-12")
]

crisis_records = []
for c_name, start_d, end_d in crises:
    sub = df_master_eval[(df_master_eval['date'] >= start_d) & (df_master_eval['date'] <= end_d)]
    if not sub.empty:
        rec = {'Crisis Epoch': c_name}
        for col in ['Strategy_Unified_News_FinBERT_Top10', 'Strategy_Baseline_News_FinBERT_Top100', 'Strategy_Baseline_Standard_Top100', 'Benchmark_SP500_Index']:
            s = sub[col]
            tot_drop = ((s.iloc[-1] / s.iloc[0]) - 1.0) * 100.0
            max_dd = (((s - s.cummax()) / s.cummax()).min()) * 100.0
            rec[f"{col}_Return"] = f"{tot_drop:+.1f}% (DD: {max_dd:.1f}%)"
        crisis_records.append(rec)

df_crisis_table = pd.DataFrame(crisis_records)
df_crisis_table.columns = ['Crisis Epoch', 'Unified Engine + News (Top 10)', 'Baseline XGBoost + News (Top 100)', 'Baseline XGBoost Standard (Top 100)', 'S&P 500 (^GSPC)']
print("=== 50-YEAR CRISIS STRESS-TESTING (10 HISTORICAL EPOCHS) ===")
print(df_crisis_table.to_string(index=False))

=== 50-YEAR CRISIS STRESS-TESTING (10 HISTORICAL EPOCHS) ===
                              Crisis Epoch Unified Engine + News (Top 10) Baseline XGBoost + News (Top 100) Baseline XGBoost Standard (Top 100)     S&P 500 (^GSPC)
   1. 1979-1982 Volcker 20% Fed Rate Shock             -8.0% (DD: -24.8%)               +24.6% (DD: -19.8%)                 +25.1% (DD: -19.7%)  +5.9% (DD: -27.1%)
                2. 1987 Black Monday Crash            -12.6% (DD: -16.9%)               -29.3% (DD: -32.8%)                 -29.2% (DD: -32.8%) -29.6% (DD: -31.5%)
    3. 1990 Gulf War Recession & Oil Shock            -25.9% (DD: -25.9%)               -24.5% (DD: -24.5%)                 -24.4% (DD: -24.4%) -19.9% (DD: -19.9%)
 4. 1997-1998 Asian Crisis & LTCM Collapse            +23.7% (DD: -15.0%)               +17.3% (DD: -20.1%)                 +17.6% (DD: -20.1%)  +7.7% (DD: -19.3%)
      5. 2000-2002 Dot-Com Bubble Collapse            -35.9% (DD: -40.2%)               -25.8% (DD: -33.4%)            

## 6. Half-Century Interactive Equity Curves & Drawdowns (1978–2026)

In [6]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=('<b>50-Year Multi-Decade Equity Curves (1978–2026 / Log Scale)</b>',
                                    '<b>Underwater Drawdown Curves (%)</b>'))

colors = {
    'Strategy_Unified_News_FinBERT_Top10': '#00CC96',
    'Strategy_Baseline_News_FinBERT_Top100': '#FF9900',
    'Strategy_Baseline_Standard_Top100': '#AB63FA',
    'Benchmark_SP500_Index': '#636EFA'
}

labels = {
    'Strategy_Unified_News_FinBERT_Top10': 'Unified Engine + Daily News FinBERT (Top 10)',
    'Strategy_Baseline_News_FinBERT_Top100': 'Baseline XGBoost + Daily News FinBERT (Top 100)',
    'Strategy_Baseline_Standard_Top100': 'Baseline Naive XGBoost Standard (Top 100)',
    'Benchmark_SP500_Index': 'S&P 500 (^GSPC Benchmark)'
}

for col, name in labels.items():
    s = df_master_eval[col]
    fig.add_trace(go.Scatter(
        x=df_master_eval['date'], y=s, name=name,
        line=dict(color=colors[col], width=3.5 if 'News' in name else 2.0)
    ), row=1, col=1)
    
    dd = ((s - s.cummax()) / s.cummax()) * 100.0
    fig.add_trace(go.Scatter(
        x=df_master_eval['date'], y=dd, name=f"{name} DD", showlegend=False,
        line=dict(color=colors[col], width=1.5)
    ), row=2, col=1)

fig.update_yaxes(type="log", row=1, col=1, title="<b>Portfolio Value ($ Log Scale)</b>")
fig.update_yaxes(row=2, col=1, title="<b>Drawdown (%)</b>")

fig.update_layout(
    template='plotly_dark', width=1150, height=800,
    title='<b>Half-Century Macro Historical Performance (1978–2026 / 12,200+ Trading Days)</b>',
    margin=dict(l=60, r=240, t=80, b=60),
    legend=dict(orientation='v', yanchor='top', y=1.0, xanchor='left', x=1.02, title=dict(text='<b>Strategy / Model</b>'))
)
fig.show()

## 7. Half-Century Annual Returns Breakdown by Year (1978–2026 Bar Chart)

In [7]:
df_master_eval['year'] = df_master_eval['date'].dt.year
strat_cols = ['Strategy_Unified_News_FinBERT_Top10', 'Strategy_Baseline_News_FinBERT_Top100', 'Strategy_Baseline_Standard_Top100', 'Benchmark_SP500_Index']

annual_records = []
for y, grp in df_master_eval.groupby('year'):
    year_label = str(y)
    start_vals = grp.iloc[0][strat_cols]
    end_vals = grp.iloc[-1][strat_cols]
    rets = ((end_vals / start_vals) - 1.0) * 100.0
    record = {'Year': year_label}
    for col in strat_cols:
        record[labels[col]] = rets[col]
    annual_records.append(record)

df_annual_table = pd.DataFrame(annual_records)
print("=== 49-YEAR ANNUAL RETURNS BREAKDOWN BY YEAR (% PROFIT: 1978-2026) ===")
print(df_annual_table.head(15).to_string(index=False))
print("...")
print(df_annual_table.tail(15).to_string(index=False))

fig_bar = go.Figure()
for col in strat_cols:
    lbl = labels[col]
    fig_bar.add_trace(go.Bar(
        x=df_annual_table['Year'],
        y=df_annual_table[lbl],
        name=lbl,
        marker_color=colors[col]
    ))

fig_bar.update_layout(
    template='plotly_dark',
    barmode='group',
    width=1200,
    height=650,
    title='<b>49 Years of Annual Returns by Year (1978–2026): Daily News FinBERT vs. Standard Baseline vs. S&P 500</b>',
    xaxis=dict(title='<b>Trading Year</b>', tickangle=-45),
    yaxis=dict(title='<b>Annual Return (%)</b>', zeroline=True, zerolinewidth=1.5, zerolinecolor='gray'),
    margin=dict(l=60, r=240, t=80, b=60),
    legend=dict(orientation='v', yanchor='top', y=1.0, xanchor='left', x=1.02, title=dict(text='<b>Strategy / Model</b>'))
)
fig_bar.show()

=== 49-YEAR ANNUAL RETURNS BREAKDOWN BY YEAR (% PROFIT: 1978-2026) ===
Year  Unified Engine + Daily News FinBERT (Top 10)  Baseline XGBoost + Daily News FinBERT (Top 100)  Baseline Naive XGBoost Standard (Top 100)  S&P 500 (^GSPC Benchmark)
1978                                      3.650540                                        16.191852                                  13.510940                   2.440845
1979                                     -0.081871                                         6.832809                                   7.947596                  11.588958
1980                                     12.816395                                        27.611183                                  24.814839                  28.366104
1981                                     -3.713625                                        -1.103352                                  -1.904509                 -10.114415
1982                                      6.060700                             

## 8. Export Results to Excel

In [8]:
output_news_path = os.path.join(LOCAL_DATA_DIR, "macro_50year_1978_2026_simulation_poc.xlsx")
with pd.ExcelWriter(output_news_path) as writer:
    df_master_eval.drop(columns=['year']).to_excel(writer, sheet_name='daily_equity_curves', index=False)
    df_analytics_table.to_excel(writer, sheet_name='summary_metrics', index=False)
    df_crisis_table.to_excel(writer, sheet_name='crisis_stress_testing', index=False)
    df_annual_table.to_excel(writer, sheet_name='annual_returns_by_year', index=False)

print(f"💾 Successfully exported 50-Year Macro Historical simulations to: {output_news_path}")

💾 Successfully exported 50-Year Macro Historical simulations to: c:\Users\honza\Desktop\projects\stock-analysis\research\notebooks\algo-alpha-execution\data\fetched\macro_50year_1978_2026_simulation_poc.xlsx
